**What changed with respect to the original file** (all flagged in the code):
1. *Reusable functions.* The original computed everything once with global variables; the backtest needs to recalibrate and reprice every day, so each step is now a function.
2. *English names.* French identifiers were renamed (`vol_implicite` -> `implied_vol`, `variance_totale` -> `total_variance`, `prix_call` -> `price_call`, `bas/haut/milieu` -> `lo/hi/mid`, ...).
3. *Implied volatility.* Same bisection, but it now returns `nan` when the price violates no-arbitrage bounds (the original silently returned a boundary value) and supports puts.
4. *Smile construction.* Out-of-the-money options (puts below the forward, calls above) with liquidity filters, instead of all calls with `bid > 0` (in-the-money calls carry little time value and give noisy vols). PCHIP is kept, with **flat extrapolation** instead of polynomial extrapolation (which can explode away from the quoted strikes).
5. *Fixing split.* $n_1=19$, $n_2=1$ derived from the roll rule (second-to-last fixing day before the Apr21 expiry of 26/02/2021) instead of the hard-coded 18/2. The Apr21 futures series in the Excel file ends on 26/02/2021, which supports this.
6. *Correlation.* Same estimator (historical correlation of log returns up to the date, fallback 0.99), but it needs at least 20 joint returns instead of 2 (a correlation computed on 2 points is always $\pm1$).
7. *Hard-coded path* replaced by a relative one; the import cell must be run first.
8. *New:* partially observed fixings (needed in February), Greeks, hedging engine, backtest, diagnostics.

**Still to verify:** the Apr21 expiry date, the default correlation, and the modelling choices listed at the end.

In [ ]:
import copy
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.optimize import brentq
from scipy.interpolate import PchipInterpolator

# ------------------------- configuration -------------------------
PARQUET = "unpivoted_bid_offer_train_clean.parquet"   # option quotes, futures settlements, SOFR
XLSX = "COJ21_COZ22_historical_series.xlsx"           # futures series (completes COJ21 up to its expiry)
TICKER_1, TICKER_2 = "COJ21 Comdty", "COK21 Comdty"   # leg 1 = April 2021 futures, leg 2 = May 2021 futures
RHO_DEFAULT = 0.99        # used when fewer than MIN_OBS joint returns are available (as in the original file)
MIN_OBS = 20
VALUATION_DATE = pd.Timestamp("2020-01-06")
START, END = "2020-01-06", "2021-02-26"               # backtest window

In [ ]:
def black_76(F, K, sigma, t, T, option_type='call'):
    # Black-76 formula for a European call or put on a futures price (interest rate r = 0).
    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * (T - t)) / (sigma * np.sqrt(T - t))
    d2 = d1 - sigma * np.sqrt(T - t)

    if option_type == 'call':
        return F * norm.cdf(d1) - K * norm.cdf(d2)
    elif option_type == 'put':
        return K * norm.cdf(-d2) - F * norm.cdf(-d1)
    else:
        raise ValueError("Invalid option type")


def black_delta(F, K, sigma, tau, option_type='call'):
    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * tau) / (sigma * np.sqrt(tau))
    return norm.cdf(d1) if option_type == 'call' else norm.cdf(d1) - 1.0


def black_vega(F, K, sigma, tau):
    d1 = (np.log(F / K) + 0.5 * sigma ** 2 * tau) / (sigma * np.sqrt(tau))
    return F * norm.pdf(d1) * np.sqrt(tau)


def implied_vol(price, F, K, T, option_type='call', tol=1e-8, max_iter=200):
    # Implied volatility by bisection. Returns nan if the price violates the no-arbitrage bounds.
    if T <= 0:
        return np.nan
    intrinsic = max(F - K, 0.0) if option_type == 'call' else max(K - F, 0.0)
    upper = F if option_type == 'call' else K
    if not (intrinsic < price < upper):
        return np.nan
    lo, hi = 0.0001, 5.0
    for _ in range(max_iter):               # safety iteration limit to prevent infinite loops
        mid = (lo + hi) / 2
        if black_76(F, K, mid, 0, T, option_type) < price:
            lo = mid
        else:
            hi = mid
        if hi - lo < tol:
            break
    return (lo + hi) / 2

In [ ]:
def load_data(parquet=PARQUET, xlsx=XLSX):
    df = pd.read_parquet(parquet)
    df["date"] = pd.to_datetime(df["date"])
    # futures settlement series for every contract
    prices = (df.drop_duplicates(subset=["date", "underlying_ticker"])
                .pivot(index="date", columns="underlying_ticker", values="underlying_settlement")
                .sort_index())
    try:                                    # complete with the Excel series (sheet "COJ21" -> ticker "COJ21 Comdty")
        for sheet in pd.ExcelFile(xlsx).sheet_names:
            s = pd.read_excel(xlsx, sheet_name=sheet)
            s["date"] = pd.to_datetime(s["date"])
            s = s.set_index("date")["settlement"]
            tk = sheet + " Comdty"
            prices[tk] = prices[tk].combine_first(s) if tk in prices.columns else s
    except FileNotFoundError:
        print("Excel file not found: futures series limited to the parquet file.")
    prices = prices.sort_index()
    sofr = df.groupby("date")["sofr_rate"].first() / 100.0          # the file is in percent
    # option quotes with the columns used below
    opts = df.rename(columns={"option_bid": "bid", "option_ask": "ask", "underlying_ticker": "ticker"}).copy()
    opts["opt_type"] = np.where(opts["option_type"].str.lower().str[0] == "c", "call", "put")
    opts["mid"] = 0.5 * (opts["bid"] + opts["ask"])
    opts["tau"] = opts["time_to_expiry_days"] / 365.0
    return df, opts, prices, sofr


def estimate_rho(prices_2, as_of, min_obs=MIN_OBS, default=RHO_DEFAULT):
    # Historical correlation of the log-returns of the two futures, using data up to `as_of` only.
    p = prices_2.loc[:pd.Timestamp(as_of)].dropna()
    returns = np.log(p / p.shift(1)).dropna()
    # Fallback to a default high correlation if the history is too short (e.g. the first valuation days)
    if len(returns) < min_obs:
        return default
    return float(np.clip(returns.iloc[:, 0].corr(returns.iloc[:, 1]), -0.999, 0.999))


df, opts, prices, sofr = load_data()
print(df.head())
print(df.dtypes)

valuation_date = VALUATION_DATE
f1 = prices.loc[valuation_date, TICKER_1]
f2 = prices.loc[valuation_date, TICKER_2]
rho = estimate_rho(prices[[TICKER_1, TICKER_2]], valuation_date)
print("Futures on the valuation date: f1 = %.2f (Apr21), f2 = %.2f (May21)" % (f1, f2))
print("Estimated correlation rho:", rho)

In [ ]:
class Smile:
    # Smile interpolated in log-moneyness ln(K/F_ref) with PCHIP; flat extrapolation; parallel shift supported.
    def __init__(self, strikes, ivs, F, shift=0.0):
        strikes = np.asarray(strikes, float)
        ivs = np.asarray(ivs, float)
        ok = np.isfinite(strikes) & np.isfinite(ivs) & (ivs > 0)
        x = np.log(strikes[ok] / F)
        v = ivs[ok]
        if len(x) == 0:
            raise ValueError("No valid smile point.")
        ux, inv = np.unique(x, return_inverse=True)
        uv = np.bincount(inv, weights=v) / np.bincount(inv)
        self.x, self.v, self.F, self.shift = ux, uv, F, shift
        self.strikes = F * np.exp(ux)
        self._interp = PchipInterpolator(ux, uv) if len(ux) >= 2 else None

    def __call__(self, K):
        x = np.log(K / self.F)
        if self._interp is None:
            base = float(self.v[0])
        else:
            base = float(self._interp(np.clip(x, self.x[0], self.x[-1])))
        return max(base + self.shift, 1e-4)

    def shifted(self, dvol):
        new = copy.copy(self)
        new.shift = self.shift + dvol
        return new


def extract_smile(day_opts, F, max_spread_ratio=0.6, min_price=0.02):
    # day_opts: quotes of one (date, ticker). Returns (Smile or None, DataFrame of the implied vols used).
    x = day_opts.dropna(subset=["bid", "ask"]).copy()
    x = x[(x["bid"] > 0) & (x["ask"] > x["bid"])]
    x = x[(x["ask"] - x["bid"]) / x["ask"] <= max_spread_ratio]
    x = x[x["mid"] >= min_price]
    otm = ((x["strike"] >= F) & (x["opt_type"] == "call")) | ((x["strike"] < F) & (x["opt_type"] == "put"))
    x = x[otm]
    x["iv"] = [implied_vol(p, F, K, T, ot) for p, K, T, ot in zip(x["mid"], x["strike"], x["tau"], x["opt_type"])]
    x = x.dropna(subset=["iv"])
    if len(x) < 3:
        return None, None      # too few points: the caller decides (e.g. reuse the previous smile)
    return Smile(x["strike"].values, x["iv"].values, F), x[["strike", "opt_type", "iv"]]


day = opts[opts["date"] == valuation_date]
smile_apr21, pts_apr = extract_smile(day[day["ticker"] == TICKER_1], f1)
smile_may21, pts_may = extract_smile(day[day["ticker"] == TICKER_2], f2)

# Comparative plot: degree-3 polynomial vs PCHIP (as in the original file)
plt.figure(figsize=(14, 6))
for i, (name, pts, sm) in enumerate([("COJ21 (April)", pts_apr, smile_apr21), ("COK21 (May)", pts_may, smile_may21)]):
    plt.subplot(1, 2, i + 1)
    k = pts["strike"].values
    grid = np.linspace(k.min(), k.max(), 100)
    plt.plot(k, pts["iv"].values, 'ko', label='Market implied vols (OTM, mid)')
    plt.plot(grid, np.polyval(np.polyfit(k, pts["iv"].values, 3), grid), 'b--', label='Degree-3 polynomial')
    plt.plot(grid, [sm(g) for g in grid], 'r-', label='PCHIP interpolation')
    plt.title('Smile comparison: ' + name)
    plt.xlabel('Strike'); plt.ylabel('Implied volatility'); plt.legend()
plt.tight_layout()
plt.show()

## Asian basket pricing (moment matching)

**Fixing schedule and roll.** Fixings are the business days of February 2021; the Apr21 contract is the reference up to and including the second-to-last fixing day before its expiry, then May21. **To verify:** `front_expiry` (26/02/2021 assumed; confirm with the ICE calendar).

**Pricing.** The remaining part of the arithmetic average is matched to a lognormal (mean and variance of the basket, from the covariance of the correlated lognormal fixings) and priced with Black-76. The original closed-form variance (`total_variance_closed_form`) is kept: it must equal the matrix version used by the generic pricer, which also handles fixings that are already observed (needed in February during the backtest, where $A=(S_{past}+\sum_{k\ \text{remaining}}f_k)/N$ and the strike becomes $K-S_{past}/N$).

The basket strike is mapped to contract-level strikes by common moneyness, $K_i=f_i\,K/\bar F$ (sticky-moneyness assumption of the original file), to read $\sigma_i$ on each smile. An optional Monte Carlo pricer (`method='mc'`, geometric control variate, antithetic variables) validates the moment matching.

In [ ]:
@dataclass
class Structure:
    fix_dates: pd.DatetimeIndex
    leg: np.ndarray            # 1 = first contract (Apr21), 2 = second (May21), for each fixing date
    notional: float = 1_000_000.0
    K_call: float = 78.0
    K_put: float = np.nan      # solved at inception (zero cost)

    @property
    def N(self):
        return len(self.fix_dates)


def build_structure(year, month, front_expiry, holidays=(), notional=1_000_000.0, K_call=78.0):
    start = pd.Timestamp(year, month, 1)
    end = start + pd.offsets.MonthEnd(0)
    days = pd.bdate_range(start, end)
    hol = [pd.Timestamp(h) for h in holidays]
    days = days[~days.isin(hol)]
    before = days[days <= pd.Timestamp(front_expiry)]
    d_star = before[-2]                        # second-to-last fixing day before expiry
    leg = np.where(days <= d_star, 1, 2)
    return Structure(days, leg, notional, K_call)


struct = build_structure(2021, 2, front_expiry="2021-02-26")
print("N =", struct.N, "| n1 =", int((struct.leg == 1).sum()), "| n2 =", int((struct.leg == 2).sum()))

In [ ]:
def _chol(S):
    jitter = 0.0
    scale = np.trace(S) / len(S)
    for _ in range(8):
        try:
            return np.linalg.cholesky(S + jitter * scale * np.eye(len(S)))
        except np.linalg.LinAlgError:
            jitter = 1e-14 if jitter == 0 else jitter * 100
    raise np.linalg.LinAlgError("Covariance matrix is not positive definite.")


def lognormal_cov(sig1, sig2, rho, theta, leg):
    # Sigma_ab = sig_a sig_b corr_ab min(theta_a, theta_b)
    sig = np.where(leg == 1, sig1, sig2)
    corr = np.where(leg[:, None] == leg[None, :], 1.0, rho)
    return np.outer(sig, sig) * corr * np.minimum.outer(theta, theta)


def simulate_correlated_futures(f1, f2, sig1, sig2, rho, theta, leg, M=100_000, seed=0, antithetic=True):
    # Simulates the fixing prices (M x n) of the two correlated futures; theta in years from t.
    theta = np.asarray(theta, float)
    leg = np.asarray(leg)
    F = np.where(leg == 1, f1, f2)
    sig = np.where(leg == 1, sig1, sig2)
    mu = -0.5 * sig ** 2 * theta
    Sigma = lognormal_cov(sig1, sig2, rho, theta, leg)
    L = _chol(Sigma)
    rng = np.random.default_rng(seed)
    n = len(theta)
    if antithetic:
        Z = rng.standard_normal((M // 2, n))
        Z = np.vstack([Z, -Z])
    else:
        Z = rng.standard_normal((M, n))
    eta = mu + Z @ L.T
    return F * np.exp(eta), dict(F=F, mu=mu, Sigma=Sigma)


def basket_moments(F, a, Sigma):
    # Mean and variance of sum_k a_k f_k (correlated lognormals).
    m1 = float(a @ F)
    cov = np.outer(F, F) * np.expm1(Sigma)
    return m1, float(a @ cov @ a)


def _black_on_forward(Fwd, K, v, D, option_type):
    d1 = (np.log(Fwd / K) + 0.5 * v ** 2) / v
    d2 = d1 - v
    if option_type == 'call':
        return D * (Fwd * norm.cdf(d1) - K * norm.cdf(d2))
    return D * (K * norm.cdf(-d2) - Fwd * norm.cdf(-d1))


def _split_fixings(struct, t, past_fix):
    t = pd.Timestamp(t)
    rem = np.asarray(struct.fix_dates > t)
    past_sum = 0.0
    for d in struct.fix_dates[~rem]:
        if past_fix is None or d not in past_fix:
            raise ValueError(f"Missing past fixing for {d.date()}")
        past_sum += past_fix[d]
    return rem, past_sum


def basket_forward(struct, t, f1, f2, past_fix=None):
    rem, past_sum = _split_fixings(struct, t, past_fix)
    F = np.where(struct.leg[rem] == 1, f1, f2)
    return past_sum / struct.N + F.sum() / struct.N


def price_asian_basket(t, struct, K, f1, f2, smile1, smile2, rho, option_type='call',
                       method='mm', past_fix=None, D=1.0, M=200_000, seed=0,
                       antithetic=True, control=True, vols=None, return_se=False):
    # Price per barrel of an Asian call/put on A = (1/N) sum_k f(t_k, contract given by the roll rule).
    t = pd.Timestamp(t)
    rem, past_sum = _split_fixings(struct, t, past_fix)
    N = struct.N
    n_rem = int(rem.sum())
    if n_rem == 0:                                           # everything already fixed
        A = past_sum / N
        pay = max(A - K, 0.0) if option_type == 'call' else max(K - A, 0.0)
        return (D * pay, 0.0) if return_se else D * pay

    leg = struct.leg[rem]
    theta = np.asarray((struct.fix_dates[rem] - t).days, float) / 365.0
    F = np.where(leg == 1, f1, f2)
    a = np.full(n_rem, 1.0 / N)
    Fbar = past_sum / N + float(a @ F)                       # forward of the full basket
    if vols is None:
        sig1 = smile1(f1 * K / Fbar)                         # equivalent strikes (common moneyness)
        sig2 = smile2(f2 * K / Fbar)
    else:
        sig1, sig2 = vols
    Keff = K - past_sum / N                                  # effective strike on the remaining part
    Sigma = lognormal_cov(sig1, sig2, rho, theta, leg)
    m1, var = basket_moments(F, a, Sigma)

    if Keff <= 0:                                            # call surely in the money / put worthless
        pay = D * (m1 - Keff) if option_type == 'call' else 0.0
        return (pay, 0.0) if return_se else pay

    if method == 'mm':
        v = np.sqrt(np.log1p(var / m1 ** 2))
        price = _black_on_forward(m1, Keff, v, D, option_type)
        return (price, 0.0) if return_se else price

    # Monte Carlo
    fix, info = simulate_correlated_futures(f1, f2, sig1, sig2, rho, theta, leg, M, seed, antithetic)
    A_rem = fix @ a
    pay_a = np.maximum(A_rem - Keff, 0.0) if option_type == 'call' else np.maximum(Keff - A_rem, 0.0)
    if control:
        s = a.sum()
        at = a / s
        mG = float(at @ (np.log(F) + info["mu"]))
        vG2 = float(at @ info["Sigma"] @ at)
        Fgeo = s * np.exp(mG + 0.5 * vG2)
        G = s * np.exp(np.log(fix) @ at)
        pay_g = np.maximum(G - Keff, 0.0) if option_type == 'call' else np.maximum(Keff - G, 0.0)
        geo_exact = _black_on_forward(Fgeo, Keff, np.sqrt(vG2), 1.0, option_type)
        x = pay_a - pay_g
        est = x.mean() + geo_exact
    else:
        x = pay_a
        est = x.mean()
    if antithetic:
        h = M // 2
        pairs = 0.5 * (x[:h] + x[h:])
        se = pairs.std(ddof=1) / np.sqrt(h)
    else:
        se = x.std(ddof=1) / np.sqrt(len(x))
    return (D * est, D * se) if return_se else D * est

In [ ]:
def total_variance_closed_form(sigma1, sigma2, theta_1, theta_2, f1, f2, rho, n1, n2):
    # Closed-form variance of the basket w1*A1 + w2*A2 (original file, with rho passed explicitly).
    N = n1 + n2
    w1, w2 = n1 / N, n2 / N
    var_A1 = sum((2 * (n1 - k) + 1) * (math.exp(sigma1**2 * theta_1[k-1]) - 1) for k in range(1, n1 + 1))
    var_A1 *= (f1**2 / n1**2)
    var_A2 = sum((2 * (n2 - j) + 1) * (math.exp(sigma2**2 * theta_2[j-1]) - 1) for j in range(1, n2 + 1))
    var_A2 *= (f2**2 / n2**2)
    cov = sum(math.exp(rho * sigma1 * sigma2 * theta_1[k-1]) - 1 for k in range(1, n1 + 1))
    cov *= (f1 * f2 / n1)
    return (w1**2 * var_A1) + (w2**2 * var_A2) + (2 * w1 * w2 * cov)

## Put strike search (bisection for the zero-cost collar)

Bisection on $K_{put}$ such that $\text{Put}(K_{put})=\text{Call}(78)$ (put too expensive $\to$ lower the strike). At each trial strike the volatilities are re-read from the smiles through the equivalent strikes.

In [ ]:
def solve_zero_cost_put(t, struct, f1, f2, smile1, smile2, rho, past_fix=None, D=1.0, method='mm', **kw):
    args = dict(f1=f1, f2=f2, smile1=smile1, smile2=smile2, rho=rho,
                method=method, past_fix=past_fix, D=D, **kw)
    price_call = price_asian_basket(t, struct, struct.K_call, option_type='call', **args)
    F_avg = basket_forward(struct, t, f1, f2, past_fix)
    lo, hi = 0.2 * F_avg, 3.0 * F_avg
    diff = lambda K: price_asian_basket(t, struct, K, option_type='put', **args) - price_call
    if diff(lo) * diff(hi) > 0:
        raise ValueError("No root in the bracket: check the call price and the smiles.")
    # Bisection loop to find K_put such that Put Price equals Call Price
    for _ in range(200):
        K_put = (lo + hi) / 2
        price_put = diff(K_put) + price_call
        if price_put > price_call:
            hi = K_put          # put is too expensive -> lower the strike
        else:
            lo = K_put          # put is too cheap -> raise the strike
        if hi - lo < 1e-8:
            break
    return (lo + hi) / 2

### Results at inception (6 January 2020)

In [ ]:
struct = build_structure(2021, 2, front_expiry="2021-02-26")
n1, n2 = int((struct.leg == 1).sum()), int((struct.leg == 2).sum())

theta_all = np.asarray((struct.fix_dates - valuation_date).days, float) / 365.0
theta_1, theta_2 = theta_all[:n1], theta_all[n1:]
F_avg = basket_forward(struct, valuation_date, f1, f2)

K_call = struct.K_call
sigma1 = smile_apr21(f1 * K_call / F_avg)
sigma2 = smile_may21(f2 * K_call / F_avg)
var_total = total_variance_closed_form(sigma1, sigma2, theta_1, theta_2, f1, f2, rho, n1, n2)
v = math.sqrt(math.log(1 + var_total / F_avg**2))

price_call = price_asian_basket(valuation_date, struct, K_call, f1, f2, smile_apr21, smile_may21, rho, 'call')
K_put = solve_zero_cost_put(valuation_date, struct, f1, f2, smile_apr21, smile_may21, rho)
price_put = price_asian_basket(valuation_date, struct, K_put, f1, f2, smile_apr21, smile_may21, rho, 'put')

# the closed-form variance of the original file must match the matrix version of the generic pricer
Sg = lognormal_cov(sigma1, sigma2, rho, theta_all, struct.leg)
_, var_generic = basket_moments(np.where(struct.leg == 1, f1, f2), np.full(struct.N, 1.0 / struct.N), Sg)
assert abs(var_total - var_generic) < 1e-9 * max(1.0, var_generic)

print("--- PRICING RESULTS (6 JANUARY 2020) ---")
print(f"n1 = {n1} (Apr21 fixings), n2 = {n2} (May21 fixings)")
print(f"Basket Forward price (F_avg): {F_avg:.2f} $")
print(f"Global basket volatility (v): {v:.4f}")
print(f"Theoretical Asian Call price at 78$: {price_call:.4f} $")
print(f"Exact Put strike for Zero-Cost Collar: {K_put:.2f} $")
print(f"Verification - Call Price: {price_call:.4f} $ | Put Price: {price_put:.4f} $")

### Validation tests

1. Black-76 / implied volatility round trip.
2. Single fixing: moment matching and Monte Carlo recover Black-76.
3. Moment matching vs Monte Carlo on the full February structure, variance reduction and AM-GM inequality.

In [ ]:
# 1) round trip
p = black_76(60., 65., 0.4, 0.0, 1.0, 'call')
assert abs(implied_vol(p, 60., 65., 1.0, 'call') - 0.4) < 1e-6
print("1) Black-76 / implied vol round trip: OK")

# 2) one fixing = Black-76
t0 = valuation_date
s1, s2 = 0.38, 0.35
one = Structure(pd.DatetimeIndex(["2021-02-26"]), np.array([1]), 1e6, 78.0)
tau = (pd.Timestamp("2021-02-26") - t0).days / 365.0
for K, typ in [(55., 'put'), (60., 'call'), (70., 'call')]:
    ref = black_76(60.0, K, s1, 0.0, tau, typ)
    mm = price_asian_basket(t0, one, K, 60.0, 60.8, None, None, 0.97, typ, 'mm', vols=(s1, s2))
    mc, se = price_asian_basket(t0, one, K, 60.0, 60.8, None, None, 0.97, typ, 'mc', vols=(s1, s2),
                                M=400_000, return_se=True)
    assert abs(mm - ref) < 1e-10 and abs(mc - ref) < 4 * se + 1e-9, (K, typ, ref, mm, mc, se)
print("2) single fixing = Black-76 (mm exact, mc within 4 SE): OK")

# 3) MM vs MC on the February structure
kw = dict(vols=(s1, s2))
mm = price_asian_basket(t0, struct, 62.0, 60.0, 60.8, None, None, 0.97, 'call', 'mm', **kw)
mc_cv, se_cv = price_asian_basket(t0, struct, 62.0, 60.0, 60.8, None, None, 0.97, 'call', 'mc',
                                  M=200_000, return_se=True, **kw)
mc_no, se_no = price_asian_basket(t0, struct, 62.0, 60.0, 60.8, None, None, 0.97, 'call', 'mc',
                                  M=200_000, control=False, return_se=True, **kw)
print("3) MM = %.4f | MC + control variate = %.4f (SE %.5f) | plain MC = %.4f (SE %.5f)" % (mm, mc_cv, se_cv, mc_no, se_no))
assert abs(mc_cv - mm) < 0.05 and se_cv < se_no
fix, _ = simulate_correlated_futures(60.0, 60.8, s1, s2, 0.97, theta_all, struct.leg, 100_000, 1)
assert np.all(fix.mean(axis=1) >= np.exp(np.log(fix).mean(axis=1)) - 1e-12)
print("   AM-GM verified on 100k paths; SE reduced by a factor %.1f" % (se_no / se_cv))

## Greeks, hedging engine and backtest

**Conventions** (to discuss in the report):
- Value of the structure for the market maker: $V=\text{Put}(K_{put})-\text{Call}(78)$ per barrel, times the notional of 1,000,000 bbl. We are long the put and short the call, so the delta is negative and the hedge buys futures.
- Futures and option premia are marked to market daily: positions have no initial value and the daily P&L is $\text{position}\times\Delta\text{price}$. No margin cost, but the **bid-ask is paid** whenever a position changes (flat half-spread on futures here; to be refined with the `underlying_bid` / `underlying_ask` columns).
- Cash earns the previous day's SOFR; transaction costs are debited from cash.
- Greeks by **bump and reprice** on the moment-matching pricer: delta per contract (central relative bump) and vega as a parallel shift of both smiles ($\pm1$ vol point).
- Optional vega hedge with **one listed option**, priced with Black-76 at the smile volatility for its strike.
- Daily recalibration: `get_state(date)` returns the futures prices, smiles, $\rho$ and SOFR of that day (falling back to the last available smile when there are too few quotes, e.g. after the Apr21 options expire on 23/02/2021).

In [ ]:
def structure_value(t, struct, state, past_fix, method='mm', D=1.0, **kw):
    # Value per barrel for the market maker (long Put(Kput), short Call(Kcall)).
    a = dict(f1=state["f1"], f2=state["f2"], smile1=state["smile1"], smile2=state["smile2"],
             rho=state["rho"], method=method, past_fix=past_fix, D=D, **kw)
    P = price_asian_basket(t, struct, struct.K_put, option_type='put', **a)
    C = price_asian_basket(t, struct, struct.K_call, option_type='call', **a)
    return P - C


def structure_greeks(t, struct, state, past_fix, bump=1e-3, dvol=0.01, **kw):
    # Delta per contract (per barrel) and vega (per unit of vol, parallel shift of both smiles).
    V = lambda st: structure_value(t, struct, st, past_fix, **kw)
    out = {}
    for key in ("f1", "f2"):
        up, dn = dict(state), dict(state)
        up[key] = state[key] * (1 + bump)
        dn[key] = state[key] * (1 - bump)
        out["delta_" + key] = (V(up) - V(dn)) / (2 * state[key] * bump)
    up, dn = dict(state), dict(state)
    up["smile1"], up["smile2"] = state["smile1"].shifted(dvol), state["smile2"].shifted(dvol)
    dn["smile1"], dn["smile2"] = state["smile1"].shifted(-dvol), state["smile2"].shifted(-dvol)
    out["vega"] = (V(up) - V(dn)) / (2 * dvol)
    return out


def listed_option(state, spec, t):
    # Price, delta, vega (per unit of vol) of a listed call/put; spec = dict(leg, K, expiry, type).
    F = state["f1"] if spec["leg"] == 1 else state["f2"]
    smile = state["smile1"] if spec["leg"] == 1 else state["smile2"]
    tau = (pd.Timestamp(spec["expiry"]) - pd.Timestamp(t)).days / 365.0
    typ = spec.get("type", "call")
    if tau <= 0:
        intr = max(F - spec["K"], 0.0) if typ == "call" else max(spec["K"] - F, 0.0)
        return intr, 0.0, 0.0
    sig = smile(spec["K"])
    return (black_76(F, spec["K"], sig, 0.0, tau, typ), black_delta(F, spec["K"], sig, tau, typ),
            black_vega(F, spec["K"], sig, tau))


def run_backtest(struct, dates, get_state, method='mm', hedge_vega=False, vega_spec=None,
                 half_spread_fut=0.01, half_spread_opt=0.02, hedge=True, **kw):
    # Daily loop: recalibration (through get_state), repricing, P&L, re-hedge.
    N = struct.notional
    past_fix, rows = {}, []
    h1 = h2 = q = 0.0                      # positions in barrels (futures) and in options
    cash, port_prev, prev = 0.0, None, None
    for i, d in enumerate(dates):
        d = pd.Timestamp(d)
        st = get_state(d)
        if d in struct.fix_dates:                                  # fixing observed today
            k = list(struct.fix_dates).index(d)
            past_fix[d] = st["f1"] if struct.leg[k] == 1 else st["f2"]
        if np.isnan(struct.K_put):                                 # inception: zero-cost strike
            struct.K_put = solve_zero_cost_put(d, struct, st["f1"], st["f2"], st["smile1"],
                                               st["smile2"], st["rho"], past_fix, method=method, **kw)
        V = N * structure_value(d, struct, st, past_fix, method, **kw)

        if prev is not None:                                       # mark-to-market of yesterday's positions
            dt = (d - prev["date"]).days / 365.0
            cash = cash * (1 + prev["sofr"] * dt)
            cash += h1 * (st["f1"] - prev["f1"]) + h2 * (st["f2"] - prev["f2"])
            if q != 0.0:
                cash += q * (listed_option(st, vega_spec, d)[0] - prev["opt"])

        if hedge:                                                  # new target positions
            g = structure_greeks(d, struct, st, past_fix, **kw)
            tq = 0.0
            o_price = o_delta = o_vega = 0.0
            if hedge_vega and vega_spec is not None and pd.Timestamp(vega_spec["expiry"]) > d:
                o_price, o_delta, o_vega = listed_option(st, vega_spec, d)
                if o_vega > 1e-8:
                    tq = -N * g["vega"] / o_vega
            t1 = -N * g["delta_f1"] - (tq * o_delta if vega_spec and vega_spec["leg"] == 1 else 0.0)
            t2 = -N * g["delta_f2"] - (tq * o_delta if vega_spec and vega_spec["leg"] == 2 else 0.0)
            cost = half_spread_fut * (abs(t1 - h1) + abs(t2 - h2)) + half_spread_opt * abs(tq - q)
            cash -= cost
            h1, h2, q = t1, t2, tq
        opt_now = listed_option(st, vega_spec, d)[0] if (vega_spec is not None and q != 0.0) else 0.0

        port = V + cash
        pnl = np.nan if port_prev is None else port - port_prev
        rows.append(dict(date=d, f1=st["f1"], f2=st["f2"], V=V, cash=cash, portfolio=port,
                         pnl=pnl, h1=h1, h2=h2, q=q))
        port_prev = port
        prev = dict(date=d, f1=st["f1"], f2=st["f2"], sofr=st["sofr"], opt=opt_now)
    return pd.DataFrame(rows).set_index("date")


def hedging_metric(res):
    # Ranking metric: mean of squared daily P&L.
    return float(np.nanmean(res["pnl"].values ** 2))

In [ ]:
def make_get_state(opts, prices, sofr, t1=TICKER_1, t2=TICKER_2, rho_default=RHO_DEFAULT):
    groups = {k: g for k, g in opts.groupby(["date", "ticker"])}
    last_smile = {t1: None, t2: None}
    cache = {}
    pair = prices[[t1, t2]]

    def smile_of(d, tk, F):
        g = groups.get((d, tk))
        out = None
        if g is not None:
            out = extract_smile(g, F)[0]
        if out is None:
            out = last_smile[tk]               # fallback: last available smile
        if out is None:
            raise ValueError(f"No smile available for {tk} on {d.date()}")
        last_smile[tk] = out
        return out

    def get_state(d):
        d = pd.Timestamp(d)
        if d in cache:
            return cache[d]
        f1_, f2_ = float(prices.loc[d, t1]), float(prices.loc[d, t2])
        st = dict(f1=f1_, f2=f2_, smile1=smile_of(d, t1, f1_), smile2=smile_of(d, t2, f2_),
                  rho=estimate_rho(pair, d, default=rho_default), sofr=float(sofr.loc[d]))
        cache[d] = st
        return st
    return get_state


get_state_real = make_get_state(opts, prices, sofr)
real_dates = prices[[TICKER_1, TICKER_2]].dropna().index
real_dates = real_dates[(real_dates >= START) & (real_dates <= END)]
print("%d business days in the backtest window" % len(real_dates))

### Backtest on real data: unhedged vs delta hedge

Metric used for the ranking: $\frac1T\sum_t P\&L_t^2$ (lowest wins).

In [ ]:
results = {}
for name, flag in [("unhedged", False), ("delta hedge", True)]:
    s = build_structure(2021, 2, front_expiry="2021-02-26")
    results[name] = run_backtest(s, real_dates, get_state_real, hedge=flag)
    r = results[name]
    print("%-12s Kput=%.2f | mean(P&L^2)=%.3e | std(P&L)=%.0f $" % (name, s.K_put, hedging_metric(r), r["pnl"].std()))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for name, r in results.items():
    ax[0].plot(r.index, r["portfolio"] - r["portfolio"].iloc[0], label=name)
    ax[1].plot(r.index, r["pnl"] ** 2, label=name)
ax[0].set_title("Portfolio value (structure + cash)"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_title("Squared daily P&L"); ax[1].set_yscale("log"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print("Worst days (delta hedge):")
print(results["delta hedge"]["pnl"].abs().sort_values(ascending=False).head(6).round(0))

### Diagnostic: correlation over time

$\rho$ as estimated on each date with the data available up to that date (historical correlation of log returns; default value for the first `MIN_OBS` returns).

In [ ]:
pair = prices[[TICKER_1, TICKER_2]]
rho_series = pd.Series({d: estimate_rho(pair, d) for d in real_dates})
plt.figure(figsize=(8, 3.2))
plt.plot(rho_series)
plt.ylim(0.9, 1.002); plt.grid(alpha=.3); plt.title("Estimated correlation Apr21 / May21")
plt.tight_layout(); plt.show()